# Week 3 – Werkcollege 5: resultaten, peer review en aftrap case

Vandaag lever je niets nieuws in. Je ziet de resultaten van Bot v2 tegen je eigen Bot v1 én tegen de rest van de klas, je test live, je beoordeelt grafieken van klasgenoten, en de docent trapt de volgende groepscase af.

## Tijdsindicatie

| Moment | Duur | Onderdeel |
|---|---|---|
| Live testen | 15 min | Bot v2 tegen klasgenoot testen |
| Resultaten: v2 vs v1 | 20 min | Toernooi ophalen, oud vs nieuw vergelijken |
| Peer review via de API | 25 min | 3 grafieken beoordelen (nu ook Plotly) |
| Aftrap groepscase | 15 min | Uitleg door docent |
| Afronden en reflectie | 15 min | Terugblik op Week 1–3 |


## Deel 1 — Live testen (15 min)

Zoek een klasgenoot op. Wissel je `mijn_bot_week3.py` uit en laat beide bots dezelfde combinaties van hand, stack en strategie zien.


In [ ]:
import sys
sys.path.append('.')

from mijn_bot_week3 import kies_actie as bot_jij
# from bot_klasgenoot_week3 import kies_actie as bot_klasgenoot

test_gevallen = [
    (["A", "A"], 1000, "tight"),
    (["7", "2"], 1000, "tight"),
    (["7", "2"], 50, "aggressive"),
]
for hand, stack, strategie in test_gevallen:
    print(hand, stack, strategie, "->", bot_jij(hand, stack, strategie))
    # print(hand, stack, strategie, "->", bot_klasgenoot(hand, stack, strategie))


## Deel 2 — Resultaten: v2 vs v1 (20 min)

De docent heeft aan het begin van dit werkcollege het toernooi vers gedraaid, met `vergelijk_met_week=1`, zodat alle inzendingen van gisteravond meetellen. Haal die uitslag op — zelfde `GET`-patroon als altijd.


In [ ]:
import requests
import pandas as pd

API_URL = "https://poker-analytics-api.onrender.com"
STUDENT_ID = "vul_hier_je_student_id_in"
TOKEN = "vul_hier_je_token_in"

response = requests.get(
    f"{API_URL}/toernooi/3",
    params={"student_id": STUDENT_ID, "vergelijk_met_week": 1},
    headers={"Authorization": f"Bearer {TOKEN}"},
)
resultaat = response.json()
resultaat["eindstand_per_bot"]


Elke bot-naam eindigt op `__w1` (je oude bot) of `__w3` (je nieuwe bot). Bouw daar een nette dataframe van: één rij per student, met een kolom voor de oude en een kolom voor de nieuwe eindstand.


In [ ]:
rijen = []
for bot_naam, eindstand in resultaat["eindstand_per_bot"].items():
    student_id, week_label = bot_naam.split("__w")
    rijen.append({"student_id": student_id, "week": int(week_label), "eindstand": eindstand})

lang_formaat = pd.DataFrame(rijen)
breed_formaat = lang_formaat.pivot(index="student_id", columns="week", values="eindstand")
breed_formaat.columns = ["eindstand_week1", "eindstand_week3"]
breed_formaat["verschil"] = breed_formaat["eindstand_week3"] - breed_formaat["eindstand_week1"]
breed_formaat.sort_values("verschil", ascending=False)


🤔 Sta jij in de plus of in de min? Is dat wat je verwachtte van je strategie-keuze?

Let op: als je dit tijdens het maken van je huiswerk al eens hebt opgehaald, kan die eerdere uitkomst een tussenstand zijn geweest — de docent draait 'm nu opnieuw met iedereens definitieve inzending.


## Deel 3 — Peer review via de API (25 min)

Zelfde recept als Week 1, met één verschil: grafieken zijn deze week vaker interactieve Plotly-grafieken in plaats van matplotlib-afbeeldingen. Dat vraagt een klein extra stukje code om te tonen.


In [ ]:
response = requests.get(
    f"{API_URL}/gallery/3",
    params={"student_id": STUDENT_ID},
    headers={"Authorization": f"Bearer {TOKEN}"},
)
gallery = response.json()
gallery


### Een grafiek bekijken (matplotlib én plotly)


In [ ]:
import base64
from IPython.display import Image, display
import plotly.graph_objects as go

eerste = gallery[0]
print(eerste["chart"]["titel"])

if eerste["chart"]["library"] == "matplotlib":
    png_bytes = base64.b64decode(eerste["chart"]["figuur_json"])
    display(Image(png_bytes))
elif eerste["chart"]["library"] == "plotly":
    fig = go.Figure(eerste["chart"]["figuur_json"])
    fig.show()


### Beoordelen en versturen

Zelfde 3 criteria als altijd: focal point, kleur & contrast, actietitel.


In [ ]:
review = {
    "week": 3,
    "anon_id": eerste["anon_id"],
    "focal_point_score": 4,                                          # pas aan
    "focal_point_opmerking": "vul hier je toelichting in",           # verplicht
    "kleur_contrast_score": 3,                                       # pas aan
    "kleur_contrast_opmerking": "vul hier je toelichting in",        # verplicht
    "actietitel_score": 5,                                           # pas aan
    "actietitel_opmerking": "vul hier je toelichting in",            # verplicht
}

response = requests.post(
    f"{API_URL}/peer-review/{STUDENT_ID}",
    json=review,
    headers={"Authorization": f"Bearer {TOKEN}"},
)
response.json()


Herhaal dit voor de overige 2 grafieken uit `gallery`.


---

## Reflectievragen

🤔 Wat is er dit werkcollege veranderd aan hoe je een grafiek bekijkt, nu er twee library's door elkaar lopen?

🎨 Was een Plotly-grafiek makkelijker of moeilijker te beoordelen op focal point dan een matplotlib-grafiek? Waarom?

💡 Keek je bij Deel 2 eerst naar je eigen verschil, of eerst naar wie er bovenaan stond? Wat zegt dat over hoe je zelf naar een leaderboard kijkt?

🤔 Je strategie-keuze werd voor het hele toernooi vastgehouden. Wat zou er veranderen als een bot halverwege een hand van strategie kon wisselen?

💡 Kijk terug op Week 1 t/m 3: welk onderdeel (Python-basis, pokerbot, of visualisatie) vond je tot nu toe het lastigst, en waarom?

---

## Deel 4 — Aftrap groepscase (15 min)

De docent introduceert de tweede groepscase.
